In [0]:
import numpy as np

from src.config import CFG
from src.functions import extract_features

# Paths
VOLUME_PATH = CFG["data"]["volume_path"]

# Load silver data
X_train = np.load(f"{VOLUME_PATH}/X_train_silver.npy")

# Test on one signal
features = extract_features(X_train[0], fs=CFG["data"]["sample_rate"])
print(f"Number of features : {len(features)}")
print(f"First 5 values     : {features[:5]}")
print(f"Any NaN            : {any(np.isnan(features))}")
print(f"Any Inf            : {any(np.isinf(features))}")

In [0]:
print("Extracting features for all train signals...")
X_train_features = np.array([
    extract_features(X_train[i], fs=CFG["data"]["sample_rate"])
    for i in range(len(X_train))
])

print(f"X_train_features shape : {X_train_features.shape}")
print(f"Any NaN                : {np.any(np.isnan(X_train_features))}")
print(f"Any Inf                : {np.any(np.isinf(X_train_features))}")

In [0]:
print("Extracting features for all test signals...")
X_test = np.load(f"{VOLUME_PATH}/X_test_silver.npy")

X_test_features = np.array([
    extract_features(X_test[i], fs=CFG["data"]["sample_rate"])
    for i in range(len(X_test))
])

print(f"X_test_features shape : {X_test_features.shape}")
print(f"Any NaN               : {np.any(np.isnan(X_test_features))}")
print(f"Any Inf               : {np.any(np.isinf(X_test_features))}")

In [0]:
feature_names = [
    # Accelerometer features
    "acc_rms_mag", "acc_rms_horizontal", "acc_peak_to_peak",
    "acc_tilt_angle_mean", "acc_tilt_angle_std", "acc_autocorr_x",
    "acc_jerk_mean", "acc_std_mag_horizontal", "acc_std_mag_3d",
    "acc_mean_abs_combined", "acc_mean_abs_horizontal",
    "acc_sum_mag", "acc_sum_horizontal_mag", "acc_velocity_estimate",
    "acc_mean_x", "acc_mean_y", "acc_mean_z",
    "acc_std_x", "acc_std_y", "acc_std_z",
    "acc_skew_x", "acc_skew_y", "acc_skew_z",
    "acc_kurt_x", "acc_kurt_y", "acc_kurt_z",
    "acc_zero_crossing_rate",
    "acc_spectral_entropy", "acc_dominant_freq", "acc_low_freq_energy",
    "acc_corr_xy", "acc_corr_yz", "acc_corr_xz",
    # Gyroscope features
    "gyro_rms_mag", "gyro_rms_horizontal", "gyro_peak_to_peak",
    "gyro_autocorr_x", "gyro_std_mag_horizontal", "gyro_std_mag_3d",
    "gyro_mean_abs_combined", "gyro_mean_abs_horizontal",
    "gyro_sum_mag", "gyro_sum_horizontal_mag",
    "gyro_mean_x", "gyro_mean_y", "gyro_mean_z",
    "gyro_std_x", "gyro_std_y", "gyro_std_z",
    "gyro_skew_x", "gyro_skew_y", "gyro_skew_z",
    "gyro_kurt_x", "gyro_kurt_y", "gyro_kurt_z",
    "gyro_zero_crossing_rate",
    "gyro_spectral_entropy", "gyro_dominant_freq", "gyro_low_freq_energy",
    "gyro_corr_xy", "gyro_corr_yz", "gyro_corr_xz",
    # Cross-sensor features
    "cross_corr_ax_gx", "cross_corr_ay_gy", "cross_corr_az_gz",
    "cross_corr_acc_mag_gyro_mag", "cross_corr_ax_gz", "cross_corr_az_gx",
]

print(f"Feature names defined : {len(feature_names)}")
assert len(feature_names) == 68, "Mismatch between feature names and feature count!"
print("Feature count matches ✅")

In [0]:
from pyspark.sql import SparkSession
import pandas as pd

spark = SparkSession.builder.getOrCreate()

# Load labels
y_train_binary     = np.load(f"{VOLUME_PATH}/y_train_binary.npy",     allow_pickle=True)
y_train_multiclass = np.load(f"{VOLUME_PATH}/y_train_multiclass.npy", allow_pickle=True)
y_test_binary      = np.load(f"{VOLUME_PATH}/y_test_binary.npy",      allow_pickle=True)
y_test_multiclass  = np.load(f"{VOLUME_PATH}/y_test_multiclass.npy",  allow_pickle=True)

# Build train feature DataFrame
train_df = pd.DataFrame(X_train_features, columns=feature_names)
train_df["label_binary"]     = y_train_binary
train_df["label_multiclass"] = y_train_multiclass
train_df["split"]            = "train"
train_df["signal_id"]        = range(len(train_df))

# Build test feature DataFrame
test_df = pd.DataFrame(X_test_features, columns=feature_names)
test_df["label_binary"]     = y_test_binary
test_df["label_multiclass"] = y_test_multiclass
test_df["split"]            = "test"
test_df["signal_id"]        = range(len(test_df))

# Save train features to Delta — Gold layer
spark.createDataFrame(train_df) \
     .write.format("delta") \
     .mode("overwrite") \
     .saveAsTable("workspace.fall_detection_project.gold_features_train")

# Save test features to Delta — Gold layer
spark.createDataFrame(test_df) \
     .write.format("delta") \
     .mode("overwrite") \
     .saveAsTable("workspace.fall_detection_project.gold_features_test")

print("Gold layer saved ✅")
print(f"Train features : {train_df.shape}")
print(f"Test features  : {test_df.shape}")
display(spark.read.table("workspace.fall_detection_project.gold_features_train").limit(5))